# InClassActivity2 — SRILM

This is a standalone Google Colab version. It does not connect to anyone's Google Drive. It builds SRILM in the current Colab runtime, asks you to upload the three corpus split files, runs the experiments, and packages the generated logs and models into a ZIP file.

**Before running:** obtain these exact input files from your course materials or the person sharing the assignment data:
- `brown-train.txt`
- `brown-dev.txt`
- `brown-test.txt`

Run the notebook from top to bottom. SRILM compilation can take a long time, and a Colab runtime reset means the setup/build cell may need to run again.

## 1. Install build tools and compile SRILM

This cell installs the basic build tools, downloads the SRILM source mirror, sets the installation path, and compiles it. It does not reuse the notebook owner's saved archive.

In [ ]:
import os
import subprocess
from pathlib import Path

# Build dependencies
subprocess.run(
    ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq build-essential git g++ make perl"],
    check=True
)

source_dir = Path("/content/srilm")
if source_dir.exists():
    subprocess.run(["rm", "-rf", str(source_dir)], check=True)

subprocess.run(
    ["git", "clone", "https://github.com/BitSpeech/SRILM.git", str(source_dir)],
    check=True
)

makefile = source_dir / "Makefile"
if makefile.exists():
    content = makefile.read_text()
    content = content.replace(
        "# SRILM = /home/speech/stolcke/project/srilm/devel",
        "SRILM = /content/srilm"
    )
    makefile.write_text(content)

print("Compiling SRILM. This may take a while...")
subprocess.run(["make", "World"], cwd=str(source_dir), check=True)

bin_dir = "/content/srilm/bin/i686-m64"
if not (Path(bin_dir) / "ngram-count").is_file():
    # Some builds use a different architecture directory; find the binaries.
    candidates = list(source_dir.glob("bin/*/ngram-count"))
    candidates_ngram = list(source_dir.glob("bin/*/ngram"))
    if not candidates or not candidates_ngram:
        raise FileNotFoundError("SRILM binaries were not found after compilation.")
    bin_dir = str(candidates[0].parent)

os.environ["PATH"] = bin_dir + ":" + os.environ["PATH"]
print("SRILM binaries:", bin_dir)
subprocess.run(["ngram-count", "-version"], check=False)
subprocess.run(["ngram", "-version"], check=False)

## 2. Upload the corpus split files

Upload the three text files together when the file picker opens. The filenames must match exactly. These are the assignment's input data, not files from the notebook owner's Drive.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
required = ["brown-train.txt", "brown-dev.txt", "brown-test.txt"]

for name in required:
    if name not in uploaded and not Path("/content", name).is_file():
        raise FileNotFoundError(
            f"Missing {name}. Upload all three required files with their exact filenames."
        )

for name in required:
    path = Path("/content") / name
    if not path.is_file():
        raise FileNotFoundError(f"Upload did not create {path}")
    print(f"{name}: {path.stat().st_size:,} bytes")

# All generated artifacts will stay in this runtime folder.
workdir = Path("/content/srilm_activity_results")
workdir.mkdir(parents=True, exist_ok=True)

## 3. Prepare training text and build/evaluate language models

This creates the sentence-marked training file and bigram count file, then builds unsmoothed bigram, Kneser–Ney bigram/trigram, and unigram/bigram interpolation models. Outputs are saved locally in `/content/srilm_activity_results`.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

bin_dir = str(Path(subprocess.check_output(["bash", "-lc", "dirname $(which ngram-count)"], text=True).strip()))
os.environ["PATH"] = bin_dir + ":" + os.environ["PATH"]

workdir = Path("/content/srilm_activity_results")
train_marked = Path("/content/brown-train-marked.txt")
with open("/content/brown-train.txt", encoding="utf-8", errors="replace") as fin, \
     open(train_marked, "w", encoding="utf-8") as fout:
    for line in fin:
        words = line.strip().split()
        if words:
            fout.write("<s> " + " ".join(words) + " </s>\\n")

shutil.copy2(train_marked, workdir / train_marked.name)
train = str(train_marked)
test = "/content/brown-test.txt"
dev = "/content/brown-dev.txt"

def run_cmd(args, label, log_file):
    log_file.write(f"\\n\\n===== {label} =====\\n")
    log_file.write("$ " + " ".join(map(str, args)) + "\\n")
    log_file.flush()
    result = subprocess.run(
        list(map(str, args)), text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    log_file.write(result.stdout)
    log_file.flush()
    print(f"{label}: {'OK' if result.returncode == 0 else 'FAILED'}")
    if result.returncode:
        print(result.stdout[-2500:])
        raise RuntimeError(f"{label} failed")
    return result.stdout

experiments = [
    ("Bigram counts",
     ["ngram-count", "-text", train, "-order", "2",
      "-write", "/content/brown-train.count"]),
    ("Unsmoothed bigram",
     ["ngram-count", "-read", "/content/brown-train.count",
      "-order", "2", "-lm", "/content/brown-bigram-unsmoothed.lm",
      "-addsmooth", "0", "-unk"]),
    ("Kneser-Ney bigram",
     ["ngram-count", "-text", train, "-order", "2",
      "-lm", "/content/brown-bigram-kn.lm", "-unk", "-kndiscount"]),
    ("Kneser-Ney trigram",
     ["ngram-count", "-text", train, "-order", "3",
      "-lm", "/content/brown-trigram-kn.lm", "-unk", "-kndiscount"]),
    ("Unigram model for interpolation",
     ["ngram-count", "-text", train, "-order", "1",
      "-lm", "/content/brown-unigram.lm", "-unk"]),
    ("Bigram model for interpolation",
     ["ngram-count", "-text", train, "-order", "2",
      "-lm", "/content/brown-bigram-interp.lm", "-unk", "-kndiscount"])
]

log_path = workdir / "experiment-results.txt"
commands_path = workdir / "experiment-commands.txt"
with open(log_path, "w", encoding="utf-8") as log, \
     open(commands_path, "w", encoding="utf-8") as cmds:
    for label, args in experiments:
        cmds.write(label + ": " + " ".join(args) + "\\n")
        run_cmd(args, label, log)

    for label, model in [
        ("Unsmoothed bigram test evaluation", "/content/brown-bigram-unsmoothed.lm"),
        ("Kneser-Ney bigram test evaluation", "/content/brown-bigram-kn.lm"),
        ("Kneser-Ney trigram test evaluation", "/content/brown-trigram-kn.lm")
    ]:
        run_cmd(["ngram", "-lm", model, "-ppl", test, "-unk"], label, log)

for name in [
    "brown-train.count", "brown-bigram-unsmoothed.lm", "brown-bigram-kn.lm",
    "brown-trigram-kn.lm", "brown-unigram.lm", "brown-bigram-interp.lm"
]:
    src = Path("/content") / name
    if src.exists():
        shutil.copy2(src, workdir / name)

print("Saved experiment log:", log_path)

## 4. Check unknown-word handling

Compare test-set output for the unsmoothed bigram model without and with `-unk`.

In [ ]:
print("Without -unk:")
subprocess.run(["ngram", "-lm", "/content/brown-bigram-unsmoothed.lm",
                "-ppl", "/content/brown-test.txt"], check=False)

print("\\nWith -unk:")
subprocess.run(["ngram", "-lm", "/content/brown-bigram-unsmoothed.lm",
                "-ppl", "/content/brown-test.txt", "-unk"], check=False)

## 5. Preprocessing: punctuation and stopword removal

This evaluates two variants: punctuation removal, and punctuation plus English stopword removal.

In [ ]:
import re
import subprocess
import shutil
from pathlib import Path

try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    stopwords = set(ENGLISH_STOP_WORDS)
except ImportError:
    subprocess.run(["pip", "-q", "install", "scikit-learn"], check=True)
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    stopwords = set(ENGLISH_STOP_WORDS)

def preprocess(source, destination, remove_stopwords=False):
    with open(source, encoding="utf-8", errors="replace") as fin, \
         open(destination, "w", encoding="utf-8") as fout:
        for line in fin:
            cleaned = re.sub(r"[^\\w\\s]", "", line)
            words = cleaned.lower().split()
            if remove_stopwords:
                words = [w for w in words if w not in stopwords]
            if words:
                fout.write("<s> " + " ".join(words) + " </s>\\n")

datasets = [("nopunct", False), ("nopunct_nostop", True)]
results_path = workdir / "preprocessing-results.txt"

with open(results_path, "w", encoding="utf-8") as log:
    for tag, remove_sw in datasets:
        for split in ["train", "test"]:
            src = f"/content/brown-{split}.txt"
            dst = f"/content/brown-{split}-{tag}.txt"
            preprocess(src, dst, remove_sw)
            shutil.copy2(dst, workdir / Path(dst).name)

        train_variant = f"/content/brown-train-{tag}.txt"
        test_variant = f"/content/brown-test-{tag}.txt"
        model = f"/content/brown-{tag}-bigram.lm"

        build = subprocess.run(
            ["ngram-count", "-text", train_variant, "-order", "2",
             "-lm", model, "-unk", "-kndiscount"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        log.write(f"\\n===== {tag}: model build =====\\n{build.stdout}\\n")
        if build.returncode:
            raise RuntimeError(f"Model build failed for {tag}")

        evaluation = subprocess.run(
            ["ngram", "-lm", model, "-ppl", test_variant, "-unk"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        log.write(f"===== {tag}: test evaluation =====\\n{evaluation.stdout}\\n")
        if evaluation.returncode:
            raise RuntimeError(f"Evaluation failed for {tag}")

        shutil.copy2(model, workdir / Path(model).name)
        print(f"{tag}: completed")
        print(evaluation.stdout)

print("Saved preprocessing results:", results_path)

## 6. Interpolation tuning on the development set

The notebook varies the interpolation weight, selects the best tested value using development-set perplexity, then evaluates that selected setting on the test set.

In [ ]:
import re
import subprocess

unigram = "/content/brown-unigram.lm"
bigram = "/content/brown-bigram-interp.lm"
dev = "/content/brown-dev.txt"
test = "/content/brown-test.txt"

weights = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
results = []
log_path = workdir / "interpolation-results.txt"

with open(log_path, "w", encoding="utf-8") as log:
    log.write("Unigram + bigram interpolation; tuning on development set\\n")
    for lam in weights:
        cmd = ["ngram", "-lm", unigram, "-mix-lm", bigram,
               "-lambda", str(lam), "-ppl", dev, "-unk"]
        result = subprocess.run(cmd, capture_output=True, text=True)
        output = result.stdout + "\\n" + result.stderr
        log.write(f"\\n===== lambda={lam} on development set =====\\n")
        log.write(" ".join(cmd) + "\\n" + output + "\\n")
        if result.returncode:
            raise RuntimeError(f"Interpolation failed for lambda={lam}:\\n{output}")
        match = re.search(r"ppl1=\\s*([0-9.]+)", output)
        if not match:
            raise RuntimeError(f"Could not parse ppl1 for lambda={lam}:\\n{output}")
        ppl1 = float(match.group(1))
        results.append((lam, ppl1))
        print(f"lambda={lam:.1f} | development ppl1={ppl1:.4f}")

best_lambda, best_dev_ppl1 = min(results, key=lambda x: x[1])
print(f"\\nSelected lambda: {best_lambda}; development ppl1: {best_dev_ppl1:.4f}")

cmd = ["ngram", "-lm", unigram, "-mix-lm", bigram,
       "-lambda", str(best_lambda), "-ppl", test, "-unk"]
result = subprocess.run(cmd, capture_output=True, text=True)
test_output = result.stdout + "\\n" + result.stderr
if result.returncode:
    raise RuntimeError(test_output)
print("\\nTEST-SET RESULTS\\n", test_output)

with open(log_path, "a", encoding="utf-8") as log:
    log.write(f"\\nSelected lambda: {best_lambda}\\n{test_output}\\n")

## 7. Tune interpolation weight and add-smoothing value

This grid search compares the listed `-addsmooth` values and interpolation weights on the development set. It then evaluates the best tested combination on the test set.

In [ ]:
import re
import subprocess

train = "/content/brown-train-marked.txt"
dev = "/content/brown-dev.txt"
test = "/content/brown-test.txt"

smoothing_values = [0.01, 0.1, 0.5, 1.0]
weights = [0.1, 0.3, 0.5, 0.7, 0.9]
results = []
log_path = workdir / "interpolation-grid-results.txt"

with open(log_path, "w", encoding="utf-8") as log:
    for smooth in smoothing_values:
        uni = f"/content/uni_add_{smooth}.lm"
        bi = f"/content/bi_add_{smooth}.lm"

        for order, model in [(1, uni), (2, bi)]:
            build = subprocess.run(
                ["ngram-count", "-text", train, "-order", str(order),
                 "-lm", model, "-addsmooth", str(smooth), "-unk"],
                capture_output=True, text=True
            )
            if build.returncode:
                raise RuntimeError(build.stdout + build.stderr)

        for lam in weights:
            cmd = ["ngram", "-lm", uni, "-mix-lm", bi,
                   "-lambda", str(lam), "-ppl", dev, "-unk"]
            run = subprocess.run(cmd, capture_output=True, text=True)
            output = run.stdout + run.stderr
            if run.returncode:
                raise RuntimeError(output)
            match = re.search(r"ppl1=\\s*([0-9.]+)", output)
            if not match:
                raise RuntimeError("Could not parse development ppl1:\\n" + output)
            ppl1 = float(match.group(1))
            results.append((ppl1, smooth, lam))
            log.write(f"addsmooth={smooth}, lambda={lam}, development_ppl1={ppl1}\\n{output}\\n")
            print(f"addsmooth={smooth:<4} lambda={lam:.1f} dev_ppl1={ppl1:.4f}")

best_ppl1, best_smooth, best_lambda = min(results)
print("\\nBest tested development configuration:")
print("addsmooth =", best_smooth)
print("lambda =", best_lambda)
print("development ppl1 =", best_ppl1)

uni = "/content/selected-unigram.lm"
bi = "/content/selected-bigram.lm"
for order, model in [(1, uni), (2, bi)]:
    build = subprocess.run(
        ["ngram-count", "-text", train, "-order", str(order),
         "-lm", model, "-addsmooth", str(best_smooth), "-unk"],
        capture_output=True, text=True
    )
    if build.returncode:
        raise RuntimeError(build.stdout + build.stderr)

cmd = ["ngram", "-lm", uni, "-mix-lm", bi,
       "-lambda", str(best_lambda), "-ppl", test, "-unk"]
run = subprocess.run(cmd, capture_output=True, text=True)
if run.returncode:
    raise RuntimeError(run.stdout + run.stderr)
test_output = run.stdout + run.stderr
print("\\nSelected configuration on test set:\\n", test_output)

with open(log_path, "a", encoding="utf-8") as log:
    log.write(f"\\nSELECTED: addsmooth={best_smooth}, lambda={best_lambda}\\n")
    log.write(f"Development ppl1={best_ppl1}\\n{test_output}\\n")
print("Saved grid results:", log_path)

## 8. Package the outputs

The models, prepared text, command log, and evaluation logs are packaged into a ZIP file for download. Nothing is written to the notebook owner's Drive.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

zip_base = "/content/InClassActivity2_SRILM_outputs"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=str(workdir))
print("Created:", zip_path)
print("Files included:")
for path in sorted(workdir.iterdir()):
    if path.is_file():
        print(f"{path.name} | {path.stat().st_size:,} bytes")

files.download(zip_path)